# Netflix Titles — Data Cleaning & Analysis

## 1. Load & First Look

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('netflix_titles.csv')

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.head(5)

In [ ]:
df.duplicated().unique()

## 2. Drop Rows With Small Amounts of Missing Data`date_added`, `rating`, and `duration` each have only a handful of missing rows (10, 4, and 3 out of 8807) — safe to drop.

In [ ]:
df = df[df['date_added'].notnull()]df = df[df['rating'].notnull()]df = df[df['duration'].notnull()]df.isnull().sum()

## 3. Explore Unique Values Per Column`director`, `cast`, and `country` are left with many missing values (2621, 825, 829) — kept as `NaN` rather than dropped, since dropping them would remove a large share of the dataset.

In [ ]:
df['show_id'].unique()

In [ ]:
df['type'].unique()

In [ ]:
df['title'].unique()

In [ ]:
df['director'].unique()

In [ ]:
df['cast'].unique()

In [ ]:
df['country'].unique()

In [ ]:
df['date_added'].unique()

In [ ]:
df['release_year'].unique()

In [ ]:
df['rating'].unique()

In [ ]:
df['duration'].unique()

In [ ]:
df['listed_in'].unique()

## 4. Clean `date_added`Some values had a leading space (e.g. `' March 31, 2017'`); `.str.strip()` removes it before converting to a real date type.

In [ ]:
df['date_added'] = df['date_added'].str.strip()df['date_added'] = pd.to_datetime(df['date_added'])df['date_added'].unique()

---# Questions

### Q1 — What is the ratio of Movies to TV Shows?

In [ ]:
df['type'].value_counts(normalize=True) * 100

### Q2 — Which country has produced the most content?`country` holds multiple comma-separated values per row, so each row is split and exploded before counting.

In [ ]:
country_list = (    df['country'].str.split(', ')    .explode()    .value_counts()    .head(10))country_list

### Q3 — Which genre (`listed_in`) is the most common?

In [ ]:
genre_count = (    df['listed_in'].str.split(', ')    .explode()    .value_counts()    .head(10))genre_count

### Q4 — What is the distribution of `rating` (content age rating)?

In [ ]:
df['rating'].value_counts(normalize=True) * 100

### Q5 — Which director has made the most titles?

In [ ]:
df['director'].str.split(', ').explode().value_counts().head(10)

### Q6 — Which actor appears in the most titles?

In [ ]:
df['cast'].str.split(', ').explode().value_counts().head(10)

### Q7 — How has the number of titles added per year changed?Sorted by year (`sort_index`), not by count, so the trend over time is actually visible.Steady growth from 2008 to 2019. The drop in 2020 lines up with COVID-related production slowdowns. The drop in 2021 is not a real decline — this dataset was collected around September 2021, so that year is only ~9 months complete.

In [ ]:
df['date_added'].dt.year.value_counts().sort_index()

### Q8 — Which `release_year` has the most titles?

In [ ]:
df.groupby('release_year').size().sort_values(ascending=False).head(10)

Breakdown by type as well — 2018 leads for Movies specifically too:

In [ ]:
df.groupby(['release_year', 'type']).size().sort_values(ascending=False).head(10)

### Q9 — Has the Movie/TV Show ratio changed over the years?

In [ ]:
df.groupby('release_year')['type'].value_counts(normalize=True).unstack() * 100

### Q10 — Do different countries focus on different genres?Both `country` and `listed_in` are multi-value columns, so both are split and exploded (independently, not paired) to get every (country, genre) combination.`.assign()` is used instead of overwriting `df['country']` directly — this keeps the original `df` untouched, so this cell is safe to re-run without corrupting anything.

In [ ]:
df_exploded = (    df.assign(        country=df['country'].str.split(', '),        listed_in=df['listed_in'].str.split(', ')    )    .explode('country')    .explode('listed_in'))df_exploded = df_exploded[df_exploded['country'] != '']country_genre = df_exploded.groupby(['country', 'listed_in']).size()top_genre_per_country = country_genre.groupby(level=0).idxmax().str[1]top_genre_per_country